In [ ]:
SELECT
    name AS contr_src_name,
    CAST(id AS STRING) AS contr_src_id,
    'WIP001' AS contr_src_sys_inst_id
FROM silver_wip_organisation

In [ ]:
%%sql
SELECT
    name AS contr_src_name,
    CAST(id AS STRING) AS contr_src_id,
    'MPB001' AS contr_src_sys_inst_id
FROM silver_drj_tenancies

In [ ]:
%%sql
SELECT DISTINCT
    rssi.src_sys_ins_organisation_short_name_conformed AS contr_src_name,
    CAST(sob.id_organisation_source AS STRING) AS contr_src_id,
    rssi.src_sys_inst_id AS contr_src_sys_inst_id
FROM silver_sone_srorganisationbranch sob
JOIN silver_rdm_source_system_instance rssi
    ON rssi.src_sys_inst_id = CONCAT('SONE', CAST(sob.id_organisation_source AS STRING))
WHERE rssi.src_sys_id = 'SONE'

In [ ]:
%%sql
SELECT DISTINCT
    rssi.src_sys_ins_organisation_short_name_conformed AS contr_src_name,
    CAST(t.service_id AS STRING) AS contr_src_id,
    rssi.src_sys_inst_id AS contr_src_sys_inst_id
FROM silver_iapt_treatments t
JOIN silver_rdm_source_system_instance rssi
    ON rssi.src_sys_inst_id = CONCAT('IAPT', CAST(t.service_id AS STRING))
WHERE rssi.src_sys_id = 'IAPT'

In [ ]:
%%sql
SELECT DISTINCT
    group_name AS contr_src_name,
    CAST(group_id AS STRING) AS contr_src_id,
    'CF001' AS contr_src_sys_inst_id
FROM silver_cf_tblcustomergroup

In [ ]:
%%sql
SELECT DISTINCT
    cf.group_name AS contr_src_name,
    CAST(cf.group_id AS STRING) AS contr_src_id,
    rssi.src_sys_inst_id AS contr_src_sys_inst_id
FROM silver_cf_tblcustomergroup cf
CROSS JOIN (
    SELECT MAX(src_sys_inst_id) AS src_sys_inst_id
    FROM silver_rdm_source_system_instance
    WHERE src_sys_id = 'CF'
      AND z_src_is_active = true
) rssig

In [ ]:
%%sql
SELECT
    name AS contr_src_name,
    CAST(id AS STRING) AS contr_src_id,
    'WIP001' AS contr_src_sys_inst_id
FROM silver_wip_organisation

UNION ALL

SELECT
    name AS contr_src_name,
    CAST(id AS STRING) AS contr_src_id,
    'MPB001' AS contr_src_sys_inst_id
FROM silver_drj_tenancies

UNION ALL

SELECT DISTINCT
    rssi.src_sys_ins_organisation_short_name_conformed AS contr_src_name,
    CAST(sob.id_organisation_source AS STRING) AS contr_src_id,
    rssi.src_sys_inst_id AS contr_src_sys_inst_id
FROM silver_sone_srorganisationbranch sob
JOIN silver_rdm_source_system_instance rssi
    ON rssi.src_sys_inst_id = CONCAT('SONE', CAST(sob.id_organisation_source AS STRING))
WHERE rssi.src_sys_id = 'SONE'

UNION ALL

SELECT DISTINCT
    rssi.src_sys_ins_organisation_short_name_conformed AS contr_src_name,
    CAST(t.service_id AS STRING) AS contr_src_id,
    rssi.src_sys_inst_id AS contr_src_sys_inst_id
FROM silver_iapt_treatments t
JOIN silver_rdm_source_system_instance rssi
    ON rssi.src_sys_inst_id = CONCAT('IAPT', CAST(t.service_id AS STRING))
WHERE rssi.src_sys_id = 'IAPT'

UNION ALL

SELECT DISTINCT
    group_name AS contr_src_name,
    CAST(group_id AS STRING) AS contr_src_id,
    'CF001' AS contr_src_sys_inst_id
FROM silver_cf_tblcustomergroup

In [ ]:
%%sql
SELECT
    contr_src_id,
    COUNT(*) AS cnt
FROM (
    SELECT
        name AS contr_src_name,
        CAST(id AS STRING) AS contr_src_id,
        'WIP001' AS contr_src_sys_inst_id
    FROM silver_wip_organisation
) x
GROUP BY contr_src_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC

In [ ]:
%%sql
SELECT COUNT(*) AS total_wip_rows
FROM silver_wip_organisation

In [ ]:
%%sql

DROP TABLE IF EXISTS silver_rdm_contract_add;

CREATE TABLE silver_rdm_contract_add AS
WITH wip_contract AS (
    -- WIP: contract source name = CustomerName, contract source id = CustomerID
    -- Source system instance is currently fixed as WIP001
    SELECT
        name AS contr_src_name,
        CAST(id AS STRING) AS contr_src_id,
        'WIP001' AS contr_src_sys_inst_id
    FROM silver_wip_organisation
),

mpb_contract AS (
    -- MPB: tenancy name and tenancy id come from silver_drj_tenancies
    -- Source system instance is currently fixed as MPB001
    SELECT
        name AS contr_src_name,
        CAST(id AS STRING) AS contr_src_id,
        'MPB001' AS contr_src_sys_inst_id
    FROM silver_drj_tenancies
),

sone_contract AS (
    -- SONE: contract source id is derived from the source organisation id
    -- Source name comes from RDM Source System Instance organisation short name
    SELECT DISTINCT
        rssi.src_sys_ins_organisation_short_name_conformed AS contr_src_name,
        CAST(sob.id_organisation_source AS STRING) AS contr_src_id,
        rssi.src_sys_inst_id AS contr_src_sys_inst_id
    FROM silver_sone_srorganisationbranch sob
    JOIN silver_rdm_source_system_instance rssi
        ON rssi.src_sys_inst_id = CONCAT('SONE', CAST(sob.id_organisation_source AS STRING))
    WHERE rssi.src_sys_id = 'SONE'
),

iapt_contract AS (
    -- IAPT: contract source id comes from service_id
    -- Source name comes from RDM Source System Instance organisation short name
    SELECT DISTINCT
        rssi.src_sys_ins_organisation_short_name_conformed AS contr_src_name,
        CAST(t.service_id AS STRING) AS contr_src_id,
        rssi.src_sys_inst_id AS contr_src_sys_inst_id
    FROM silver_iapt_treatments t
    JOIN silver_rdm_source_system_instance rssi
        ON rssi.src_sys_inst_id = CONCAT('IAPT', CAST(t.service_id AS STRING))
    WHERE rssi.src_sys_id = 'IAPT'
),

cf_contract AS (
    -- CF: contract source name = group_name, contract source id = group_id
    -- Source system instance is currently fixed as CF001
    SELECT DISTINCT
        group_name AS contr_src_name,
        CAST(group_id AS STRING) AS contr_src_id,
        'CF001' AS contr_src_sys_inst_id
    FROM silver_cf_tblcustomergroup
),

all_contract_candidates AS (
    SELECT * FROM wip_contract
    UNION ALL
    SELECT * FROM mpb_contract
    UNION ALL
    SELECT * FROM sone_contract
    UNION ALL
    SELECT * FROM iapt_contract
    UNION ALL
    SELECT * FROM cf_contract
)

SELECT DISTINCT
    src.contr_src_name,
    src.contr_src_id,
    src.contr_src_sys_inst_id
FROM all_contract_candidates src
LEFT JOIN silver_rdm_contract tgt
    ON TRIM(LOWER(src.contr_src_name)) = TRIM(LOWER(tgt.contr_src_name))
   AND TRIM(LOWER(src.contr_src_id)) = TRIM(LOWER(tgt.contr_src_id))
   AND TRIM(LOWER(src.contr_src_sys_inst_id)) = TRIM(LOWER(tgt.contr_src_sys_inst_id))
WHERE tgt.contr_src_id IS NULL;

In [ ]:
%%sql
SELECT COUNT(*) AS total_new_rows
FROM silver_rdm_contract_add;

In [ ]:
%%sql
SELECT *
FROM silver_rdm_contract_add
ORDER BY contr_src_sys_inst_id, contr_src_id
LIMIT 100;

In [ ]:
%%sql
SELECT
    a.*
FROM silver_rdm_contract_add a
JOIN silver_rdm_contract b
    ON TRIM(LOWER(a.contr_src_name)) = TRIM(LOWER(b.contr_src_name))
   AND TRIM(LOWER(a.contr_src_id)) = TRIM(LOWER(b.contr_src_id))
   AND TRIM(LOWER(a.contr_src_sys_inst_id)) = TRIM(LOWER(b.contr_src_sys_inst_id));

In [ ]:
WIP
Added RDM Contract add logic for WIP.
Mapped contr_src_name from WIP organisation/customer name and contr_src_id from WIP organisation/customer id.
Used WIP001 as the source system instance id.
Only new rows not already present in silver_rdm_contract are included.
Ready for UAT.

MPB
Added RDM Contract add logic for MPB.
Mapped contr_src_name from tenancy name and contr_src_id from tenancy id in silver_drj_tenancies.
Used MPB001 as the source system instance id.
Only new rows not already present in silver_rdm_contract are included.
Ready for UAT.

CF
Added RDM Contract add logic for CF.
Mapped contr_src_name from group_name and contr_src_id from group_id in silver_cf_tblcustomergroup.
Used CF001 as the source system instance id.
Only new rows not already present in silver_rdm_contract are included.
Ready for UAT.

IAPTUS
Added RDM Contract add logic for IAPTUS.
Mapped contr_src_id from service_id in silver_iapt_treatments.
Mapped contr_src_name from the organisation short name in silver_rdm_source_system_instance.
Used the matching IAPT source system instance id.
Only new rows not already present in silver_rdm_contract are included.

SystemOne
Added RDM Contract add logic for SystemOne.
Mapped contr_src_id from the SystemOne source instance suffix.
Mapped contr_src_name from the organisation short name in silver_rdm_source_system_instance.
Used the matching SystemOne source system instance id.
Only new rows not already present in silver_rdm_contract are included.